# 3.1 - Clasification Model: Training and Evaluation

## 1: Dependencies and Imports

In [1]:
# Instalar LightGBM directamente desde el notebook
%pip install lightgbm optuna flaml

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Standard library
import os

# Third-party
import joblib
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import seaborn as sns
from flaml import AutoML
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, train_test_split

## 2: Load Data

In [3]:
# Path
input_file_path = '../data/Filtered.pkl'

if os.path.exists(input_file_path):
    df = pd.read_pickle(input_file_path)
    rows, cols = df.shape
    print(f"Dataset successfully loaded: {rows} rows and {cols} columns.")
else:
    print(f"Error: File not found at {input_file_path}")

# Quick preview of the loaded data
display(df.head(3))

Dataset successfully loaded: 114567 rows and 16 columns.


,order_id,is_delayed,actual_delivery_days,estimated_delivery_margin_days,purchase_month,purchase_day_of_week,product_weight_g,product_volume_cm3,product_category_name_english,customer_zip_code_prefix,seller_zip_code_prefix,is_same_state,customer_state_num_pred,seller_state_num_pred,freight_value,price
0,e481f51cbdc54678b7cc49136f2d6af7,0,8.0,15,10,0,500.0,1976.0,housewares,3149,9350.0,1,25,25,8.72,29.99
1,e481f51cbdc54678b7cc49136f2d6af7,0,8.0,15,10,0,500.0,1976.0,housewares,3149,9350.0,1,25,25,8.72,29.99
2,e481f51cbdc54678b7cc49136f2d6af7,0,8.0,15,10,0,500.0,1976.0,housewares,3149,9350.0,1,25,25,8.72,29.99


## 3: Data Preparation

In [4]:
# Feature Selection
drop_columns = ['order_id', 'customer_lat', 'customer_lng', 'seller_lat', 'seller_lng', 'customer_zip_code_prefix', 'seller_zip_code_prefix', 'actual_delivery_days']
X = df.drop(columns=drop_columns + ['is_delayed'], errors='ignore')
y = df['is_delayed']

# Categorical Feature Definintion
categorical_features = [
    'product_category_name_english', 
    'customer_state_num_pred', 
    'seller_state_num_pred',
    'purchase_month', 
    'purchase_day_of_week',
    'is_same_state'
]

for feature in categorical_features:
    if feature in X.columns:
        X[feature] = X[feature].astype('category')

print(f"Encoding completed. Number of predictor variables: {X.shape[1]}")
print(f"Categorical features identified: {categorical_features}")

Encoding completed. Number of predictor variables: 11
Categorical features identified: ['product_category_name_english', 'customer_state_num_pred', 'seller_state_num_pred', 'purchase_month', 'purchase_day_of_week', 'is_same_state']


In [5]:
# Calculate the frequency of each class in the target variable
negative_count = (y == 0).sum()
positive_count = (y == 1).sum()

# Calculate the recommended scale_pos_weight
# This ratio helps the model penalize misclassifications of the minority class more heavily
scale_pos_weight = negative_count / positive_count

print("Class Balance Analysis:")
print(f" - On-time orders (0): {negative_count}")
print(f" - Delayed orders (1): {positive_count}")
print(f" - Recommended 'scale_pos_weight': {scale_pos_weight:.2f}")

print("\nThis value will be utilized as a hyperparameter to compensate for the minority class imbalance.")

Class Balance Analysis:
 - On-time orders (0): 105642
 - Delayed orders (1): 8925
 - Recommended 'scale_pos_weight': 11.84

This value will be utilized as a hyperparameter to compensate for the minority class imbalance.


In [6]:
# Initial split: Reserve 15% for the final Hold-out Test set
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=42
)

# Second split: Divide the remaining 85% into Training and Validation sets
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.15, stratify=y_train_val, random_state=42
)

print("Data partitioning complete (approx. 72% / 13% / 15%):")
print(f" - Training Set (Train):   {X_train.shape[0]} records")
print(f" - Validation Set (Val):   {X_val.shape[0]} records")
print(f" - Final Test Set (Test):  {X_test.shape[0]} records")

Data partitioning complete (approx. 72% / 13% / 15%):
 - Training Set (Train):   82773 records
 - Validation Set (Val):   14608 records
 - Final Test Set (Test):  17186 records


## 4: Hyperparam Optimization

### 4.1: Optuna

In [7]:
# def objective(trial):
#     # Optimized search space for better speed/performance balance
#     params = {
#         'objective': 'binary',
#         'metric': 'auc',
#         'verbosity': -1,
#         'boosting_type': 'gbdt',
#         'random_state': 42,
#         'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
#         'n_estimators': 2000, # Sufficient for most tasks with early stopping
#         'num_leaves': trial.suggest_int('num_leaves', 20, 128), # Reduced from 512 to speed up training
#         'max_depth': trial.suggest_int('max_depth', 3, 12),
#         'min_child_samples': trial.suggest_int('min_child_samples', 20, 100),
#         'subsample': trial.suggest_float('subsample', 0.6, 1.0),
#         'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
#         'lambda_l1': trial.suggest_float('lambda_l1', 1e-5, 1.0, log=True),
#         'lambda_l2': trial.suggest_float('lambda_l2', 1e-5, 1.0, log=True),
#         'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1.0, 5.0) 
#     }

#     # 3-Fold CV to reduce total training time by 40%
#     cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
#     cv_scores = []

#     for train_idx, val_idx in cv.split(X_train_val, y_train_val):
#         X_t, X_v = X_train_val.iloc[train_idx], X_train_val.iloc[val_idx]
#         y_t, y_v = y_train_val.iloc[train_idx], y_train_val.iloc[val_idx]

#         model = lgb.LGBMClassifier(**params)
#         model.fit(
#             X_t, y_t,
#             eval_set=[(X_v, y_v)],
#             categorical_feature=categorical_features,
#             callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
#         )

#         preds = model.predict(X_v)
#         cv_scores.append(f1_score(y_v, preds))

#     return np.mean(cv_scores)

In [8]:
# # Bayesian Study
# study = optuna.create_study(
#     direction='maximize',
#     sampler=optuna.samplers.TPESampler(multivariate=True, seed=42)
# )

# # Reduced trials to 50 for a faster but still effective search
# study.optimize(objective, n_trials=50, show_progress_bar=True)

# print(f"\nOptimal CV F1-Score: {study.best_value:.4f}")
# print("Best Configuration:", study.best_params)

# best_params_optuna = study.best_params

In [9]:
def objective(trial):
    # Aggressive search space for high-capacity models
    params = {
        'objective': 'binary',
        'metric': 'auc',
        'verbosity': -1,
        'boosting_type': 'gbdt',
        'random_state': 42,
        'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.2, log=True),
        'n_estimators': 3000,
        'num_leaves': trial.suggest_int('num_leaves', 8, 512),
        'max_depth': trial.suggest_int('max_depth', -1, 20),
        'min_child_samples': trial.suggest_int('min_child_samples', 2, 150),
        'subsample': trial.suggest_float('subsample', 0.4, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'lambda_l1': trial.suggest_float('lambda_l1', 1e-8, 10.0, log=True),
        'lambda_l2': trial.suggest_float('lambda_l2', 1e-8, 10.0, log=True),
        'min_split_gain': trial.suggest_float('min_split_gain', 0, 1.0),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1.0, 10.0) 
    }

    # 5-Fold Stratified CV for robust F1 estimation
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = []

    for train_idx, val_idx in cv.split(X_train_val, y_train_val):
        X_t, X_v = X_train_val.iloc[train_idx], X_train_val.iloc[val_idx]
        y_t, y_v = y_train_val.iloc[train_idx], y_train_val.iloc[val_idx]

        model = lgb.LGBMClassifier(**params)
        model.fit(
            X_t, y_t,
            eval_set=[(X_v, y_v)],
            categorical_feature=categorical_features,
            callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)]
        )

        # Optimization target: Mean F1-Score
        preds = model.predict(X_v)
        cv_scores.append(f1_score(y_v, preds))

    return np.mean(cv_scores)


In [ ]:
# Bayesian Study with Multivariate TPE for parameter correlation detection
study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(multivariate=True, seed=42)
)

# Increased trials to explore the aggressive space
study.optimize(objective, n_trials=150, show_progress_bar=True)

print(f"\nOptimal CV F1-Score: {study.best_value:.4f}")
print("Best Configuration:", study.best_params)

best_params_optuna = study.best_params

/var/folders/r_/f4xbyqj91qq69g9k28thxf080000gn/T/ipykernel_7558/2049394476.py:4: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  sampler=optuna.samplers.TPESampler(multivariate=True, seed=42)
[I 2026-04-13 23:18:25,375] A new study created in memory with name: no-name-7316a495-d693-4f95-b710-72c7c98a070d


  0%|          | 0/150 [00:00<?, ?it/s]

### 4.2: Auto ML

In [ ]:
# Initialize AutoML engine
automl = AutoML()

# Aggressive configuration for high-confidence optimization
automl_settings = {
    "time_budget": 600,
    "metric": 'f1',
    "task": 'classification',
    "estimator_list": ['lgbm'],
    "eval_method": 'cv',
    "n_splits": 10,
    "seed": 42,
    "verbose": 0
}

In [ ]:
# Run optimization process
automl.fit(
    X_train=X_train, 
    y_train=y_train, 
    **automl_settings
)

# Output optimization results
print(f"Best Hyperparameter Config: {automl.best_config}")
print(f"Best Validation RMSE: {automl.best_loss:.4f}")

# Extract the best hyperparameters dictionary
best_params_automl = automl.best_config

## 5: Train Model with Optimized Parameters

TESTTESTTEST

In [ ]:
best_params = best_params_optuna

In [ ]:
# Retrieve and consolidate optimal hyperparameters
final_params = best_params.copy()
final_params.update({
    'objective': 'binary',
    'metric': 'auc',
    'n_estimators': 2000,
    'random_state': 42,
    'n_jobs': -1,
    'verbosity': -1
})

# Initialize the final classifier
lgbm_model = lgb.LGBMClassifier(**final_params)

In [ ]:
# Fit model
lgbm_model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_val, y_val)],
    eval_names=['Training', 'Validation'],
    categorical_feature=categorical_features,
    callbacks=[
        lgb.early_stopping(stopping_rounds=100, first_metric_only=True, verbose=True),
        lgb.log_evaluation(period=100)
    ]
)

print("\nFinal optimized model is trained and ready for performance evaluation.")

In [ ]:
# Generate hard predictions (binary: 0 or 1) and soft probabilities (range: 0.0 to 1.0)
y_pred = lgbm_model.predict(X_test)
y_prob = lgbm_model.predict_proba(X_test)[:, 1]

# Compute core performance metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

# Display Executive Summary
print(f"Accuracy:  {accuracy:.4f} (Note: May be misleading in imbalanced contexts)")
print(f"Precision: {precision:.4f} (True Positives / Total Predicted Positives)")
print(f"Recall:    {recall:.4f} (True Positives / Total Actual Positives)")
print(f"F1-Score:  {f1:.4f} (Harmonic Mean of Precision and Recall)")
print(f"ROC-AUC:   {roc_auc:.4f} (Model's discriminative capability)")

In [ ]:
# Generating a granular breakdown of metrics per class
print(classification_report(
    y_test, 
    y_pred, 
    target_names=['On-Time (0)', 'Delayed (1)']
))

In [ ]:
# Confusion Matrix Visualization

plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_test, y_pred)

# Initialize the heatmap visualization
# We use 'fmt=d' to display raw integers and 'Blues' for a clean, analytical look
sns.heatmap(
    cm, 
    annot=True, 
    fmt='d', 
    cmap='Blues', 
    xticklabels=['Pred. On-Time', 'Pred. Delayed'], 
    yticklabels=['Actual On-Time', 'Actual Delayed'],
    annot_kws={"size": 14, "weight": "bold"}
)

plt.title('Confusion Matrix: LightGBM Logistics Model', fontsize=16, fontweight='bold', pad=15)
plt.ylabel('Ground Truth (Actual)', fontsize=12, fontweight='bold')
plt.xlabel('Model Prediction', fontsize=12, fontweight='bold')
plt.show()

END TESTTESTTEST

In [ ]:
best_params = best_params_optuna
best_params = best_params_automl

In [ ]:
# Retrieve and consolidate optimal hyperparameters
final_params = best_params.copy()
final_params.update({
    'objective': 'binary',
    'metric': 'auc',
    'n_estimators': 2000,
    'random_state': 42,
    'n_jobs': -1,
    'verbosity': -1
})

# Initialize the final classifier
lgbm_model = lgb.LGBMClassifier(**final_params)

In [ ]:
# Fit model
lgbm_model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_val, y_val)],
    eval_names=['Training', 'Validation'],
    categorical_feature=categorical_features,
    callbacks=[
        lgb.early_stopping(stopping_rounds=100, first_metric_only=True, verbose=True),
        lgb.log_evaluation(period=100)
    ]
)

print("\nFinal optimized model is trained and ready for performance evaluation.")

## 6: Result Calculation

In [ ]:
# Generate hard predictions (binary: 0 or 1) and soft probabilities (range: 0.0 to 1.0)
y_pred = lgbm_model.predict(X_test)
y_prob = lgbm_model.predict_proba(X_test)[:, 1]

# Compute core performance metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

# Display Executive Summary
print(f"Accuracy:  {accuracy:.4f} (Note: May be misleading in imbalanced contexts)")
print(f"Precision: {precision:.4f} (True Positives / Total Predicted Positives)")
print(f"Recall:    {recall:.4f} (True Positives / Total Actual Positives)")
print(f"F1-Score:  {f1:.4f} (Harmonic Mean of Precision and Recall)")
print(f"ROC-AUC:   {roc_auc:.4f} (Model's discriminative capability)")

In [ ]:
# Generating a granular breakdown of metrics per class
print(classification_report(
    y_test, 
    y_pred, 
    target_names=['On-Time (0)', 'Delayed (1)']
))

In [ ]:
# Confusion Matrix Visualization

plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_test, y_pred)

# Initialize the heatmap visualization
# We use 'fmt=d' to display raw integers and 'Blues' for a clean, analytical look
sns.heatmap(
    cm, 
    annot=True, 
    fmt='d', 
    cmap='Blues', 
    xticklabels=['Pred. On-Time', 'Pred. Delayed'], 
    yticklabels=['Actual On-Time', 'Actual Delayed'],
    annot_kws={"size": 14, "weight": "bold"}
)

plt.title('Confusion Matrix: LightGBM Logistics Model', fontsize=16, fontweight='bold', pad=15)
plt.ylabel('Ground Truth (Actual)', fontsize=12, fontweight='bold')
plt.xlabel('Model Prediction', fontsize=12, fontweight='bold')
plt.show()

## 7: Save Model

In [ ]:
# Ensure the directory exists
models_dir = '../models'
os.makedirs(models_dir, exist_ok=True)

# Define the model path; joblib is often more efficient for large NumPy arrays
model_file_path = os.path.join(models_dir, 'lgbm_logistics_model_opt.joblib')

# Serialize the model
joblib.dump(lgbm_model, model_file_path)

print(f"Model successfully saved to: {model_file_path}")